In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, optimizers
import os

# 1. Configuration
BATCH_SIZE = 32
IMG_SIZE = (128, 128)  # Keep small for speed
EPOCHS = 10
DATA_DIR = '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake'

# 2. Data Generators (Split data automatically)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'train'),  # Using train dir for split to save RAM
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

# 3. Model Architecture (Transfer Learning)
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
base_model.trainable = False  # Freeze base initially

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')  # Binary output: 0=Fake, 1=Real
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 4. Train
history = model.fit(train_generator, validation_data=val_generator, epochs=EPOCHS)

# 5. Save Model (with include_optimizer=False for better compatibility)
model.save('deepfake_image_model.h5', include_optimizer=False)
print("Model saved! Download this file.")
print("Class Indices:", train_generator.class_indices)

In [ ]:
# Export models in multiple formats for serving compatibility
import tensorflow as tf
try:
    import keras as k3
    keras_version = k3.__version__
except Exception:
    k3 = None
    keras_version = 'not-installed'
print('TF version:', tf.__version__)
print('Keras (standalone) version:', keras_version)

# 1) H5 (no optimizer) – good for tf.keras.load_model(..., compile=False)
model.save('deepfake_image_model.h5', include_optimizer=False)
print('Saved H5 -> deepfake_image_model.h5')

# 2) Keras v3 native format (.keras) – recommended if using standalone keras>=3
try:
    model.save('deepfake_image_model.keras', include_optimizer=False)
    print('Saved Keras format -> deepfake_image_model.keras')
except Exception as e:
    print('Saving .keras format failed:', e)

# 3) TensorFlow SavedModel directory – robust across TF versions
try:
    model.save('deepfake_image_model_saved', save_format='tf')
    print('Saved SavedModel dir -> deepfake_image_model_saved/')
except Exception as e:
    print('Saving SavedModel format failed:', e)

# Tip: upload one of these to backend/models and restart the backend.